The goal of this notebook is to showcase


1.   How to load and preprocess the classification challenge data
2.   How to build a baseline model
3.   How to prepare results to submit to the kaggle challenge



## Loading necessary packages

In [2]:
import os
import pickle
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

from google.colab import drive

Mounting Google Drive space for data access:

- You need to download the files train_X_y.pkl and test_X.pkl from [kaggle challenge site](https://www.kaggle.com/t/05ca47eb65b8426e9d53a2cc90a5875d), and then upload them to your **base_path** directory.

In [3]:
drive.mount('/content/drive')
base_path = 'drive/MyDrive/image-classification-challenge-w-26/W2026-0-0.3/'



Mounted at /content/drive


## Loading Classification Challenge data

In [4]:
# Training data
with open(os.path.join(base_path, 'train_X_y.pkl'),'rb') as f:
    X_train, y_train = pickle.load(f)

In [5]:
# Test data
with open(os.path.join(base_path, 'test_X.pkl'),'rb') as f:
    X_test = pickle.load(f)

In [6]:
X_train.shape, X_test.shape

((42000, 16, 16, 3), (18000, 16, 16, 3))

## Data preprocessing

In [7]:
X_train.min(), X_train.max()

(np.uint8(0), np.uint8(255))

In [8]:
# Data normalization
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

In [9]:
X_train.min(), X_train.max()

(np.float32(0.0), np.float32(1.0))

In [10]:
# transforming to 1D vector
X_train = X_train.reshape(X_train.shape[0], -1)
y_train = y_train.flatten()
X_test = X_test.reshape(X_test.shape[0], -1)

## Training a Baseline Random Forest model

In [ ]:
baseline_model = RandomForestClassifier()
baseline_model.fit(X_train, y_train)

## Making predictions and preparing submission file for the kaggle challenge

In [ ]:
# Making predictions
y_predict = baseline_model.predict(X_test)

In [ ]:
# Pack results in a pandas dataframe
test_predictions = pd.DataFrame({
    'rowId': range(0, len(y_predict.flatten())),
    'label': y_predict.flatten()})

In [ ]:
test_predictions.head()

,rowId,label
0,0,6
1,1,9
2,2,8
3,3,1
4,4,2


In [ ]:
# Exporting results
test_predictions.to_csv(os.path.join(base_path, 'sample_submission.csv'), index=False)

- HPC Cluster
-

Once you have successfully generated your results, say, "my_baseline_results.csv", you can upload this file for the kaggle challenge results.

## CNN Pipeline (Refactored, End-to-End)
This single pipeline cell replaces the fragmented CNN section and keeps the notebook flow smooth after the Random Forest baseline.

What it does:
- Reshapes image tensors for CNN input (`N, C, H, W`)
- Builds a stronger CNN with BatchNorm + Dropout
- Uses a train/validation split to compute a score before export
- Trains with light augmentation (random horizontal flip) and scheduler
- Exports a Kaggle submission only if validation score is above a threshold

In [ ]:
import os
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from torch.utils.data import Dataset, DataLoader

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# ---------- Data prep: flattened -> image tensors ----------
X_train_img = X_train.reshape(-1, 16, 16, 3).transpose(0, 3, 1, 2).astype('float32')
X_test_img = X_test.reshape(-1, 16, 16, 3).transpose(0, 3, 1, 2).astype('float32')
y_train_1d = y_train.reshape(-1).astype('int64')

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_img,
    y_train_1d,
    test_size=0.15,
    random_state=SEED,
    stratify=y_train_1d
)

class ImageDataset(Dataset):
    def __init__(self, images, labels=None, augment=False):
        self.images = images
        self.labels = labels
        self.augment = augment

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        x = torch.from_numpy(self.images[idx]).float()
        if self.augment and torch.rand(1).item() < 0.5:
            x = torch.flip(x, dims=[2])  # horizontal flip
        if self.labels is None:
            return x
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y

train_ds = ImageDataset(X_tr, y_tr, augment=True)
val_ds = ImageDataset(X_val, y_val, augment=False)
test_ds = ImageDataset(X_test_img, labels=None, augment=False)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())

# ---------- Improved CNN ----------
class BetterCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(0.15),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(0.20),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.35),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = BetterCNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
epochs = 25
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

# ---------- Training + validation ----------
best_val_acc = 0.0
best_state = None

for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xb.size(0)

    model.eval()
    val_preds, val_true = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            logits = model(xb)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            val_preds.extend(preds.tolist())
            val_true.extend(yb.numpy().tolist())

    val_acc = accuracy_score(val_true, val_preds)
    epoch_loss = running_loss / len(train_loader.dataset)
    scheduler.step()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())

    print(f'Epoch {epoch:02d}/{epochs} | Train Loss: {epoch_loss:.4f} | Val Acc: {val_acc:.4f}')

# Restore best model
if best_state is not None:
    model.load_state_dict(best_state)

# ---------- Score check before submission export ----------
print(f'Best validation accuracy: {best_val_acc:.4f}')
MIN_REQUIRED_SCORE = 0.6500  # target > 0.643 baseline public score
if best_val_acc >= MIN_REQUIRED_SCORE:
    model.eval()
    test_preds = []
    with torch.no_grad():
        for xb in test_loader:
            xb = xb.to(device)
            logits = model(xb)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            test_preds.extend(preds.tolist())

    submission = pd.DataFrame({
        'rowId': np.arange(len(test_preds)),
        'label': np.array(test_preds, dtype=np.int64)
    })
    submission_path = os.path.join(base_path, 'cnn_submission_improved.csv')
    submission.to_csv(submission_path, index=False)
    print(f'Validation gate passed ({best_val_acc:.4f} >= {MIN_REQUIRED_SCORE:.4f}).')
    print(f'Submission saved to: {submission_path}')
    display(submission.head())
else:
    print(f'Validation gate not passed ({best_val_acc:.4f} < {MIN_REQUIRED_SCORE:.4f}).')
    print('Submission CSV was not exported. Adjust model/training and re-run.')